# MapWorkIds — location_work_ids maintenance (oxjob #764)

Nightly identity step, runs between `Locations_with_Types` and `Locations_Mapped`.
Resolves or mints a work_id for every anchor `(provenance, namespace, native_id)`
not yet pinned in the registry. Insert-only for existing anchors: a non-NULL
registry work_id is never changed here — repoints are explicit sweep operations.
`work_id_map` keeps clustering + minting; consumers read only its frozen `work_id`
column (`COALESCE(paper_id, id)`; paper_id adoption retired — oxjob #764).

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.work_id_map') (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 6600000001 INCREMENT BY 1),
  paper_id STRING,
  doi STRING,
  pmid STRING,
  arxiv STRING,
  title_author STRING,
  work_id_source STRING,
  published_date DATE,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  work_id BIGINT
)
CLUSTER BY (doi, pmid, arxiv, title_author)
TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = '60 days',
  'delta.logRetentionDuration' = '60 days',
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.location_work_ids') (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  work_id BIGINT,
  work_id_source STRING,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  seeded_dt DATE,
  seeded_from STRING
)
CLUSTER BY (provenance, native_id_namespace, native_id)

## Pending anchors → mint candidates
Pending = w_types anchors with no registry entry, plus registry entries still NULL
(NULL anchors retry nightly until they pin).

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
CLUSTER BY (doi, pmid, arxiv, title_author)
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
  QUALIFY rwcnt = 1
),
pending AS (
  SELECT t.*
  FROM t
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  WHERE r.native_id IS NULL OR r.work_id IS NULL
)
SELECT
  merge_key.doi           AS doi,
  merge_key.pmid          AS pmid,
  merge_key.arxiv         AS arxiv,
  merge_key.title_author  AS title_author,
  (
    IF(merge_key.doi IS NOT NULL, 4, 0) +
    IF(merge_key.pmid IS NOT NULL, 3, 0) +
    IF(merge_key.arxiv IS NOT NULL, 2, 0) +
    IF(merge_key.title_author IS NOT NULL, 1, 0)
  ) AS key_score,
  MIN(published_date) AS published_date,
  current_date() AS openalex_created_dt,
  current_timestamp() AS openalex_updated_dt
FROM pending
GROUP BY doi, pmid, arxiv, title_author

### Mint into `work_id_map` — `DOI` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT regexp_replace(doi, '[^a-zA-Z0-9\./-]', '') as cleaned_doi, *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY regexp_replace(doi, '[^a-zA-Z0-9\./-]', '')
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON regexp_replace(target.doi, '[^a-zA-Z0-9\./-]', '') = cleaned_doi
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'doi'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'doi'
)

### Mint into `work_id_map` — `PMID` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY pmid
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.pmid = source.pmid
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'pmid'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'pmid'
)

### Mint into `work_id_map` — `ARXIV` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NULL AND arxiv IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY arxiv
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.arxiv = source.arxiv
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'arxiv'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'arxiv'
)

### Mint into `work_id_map` — `TITLE_AUTHOR` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NULL AND arxiv IS NULL AND title_author IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY title_author
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.title_author = source.title_author
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'title_author'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'title_author'
)

In [0]:
-- Frozen work_id for freshly minted rows (paper_id adoption is retired, #764)
UPDATE identifier('openalex' || :env_suffix || '.works.work_id_map')
SET work_id = COALESCE(CAST(paper_id AS BIGINT), id)
WHERE work_id IS NULL

## Resolve pending anchors and upsert the registry
Tier order doi → pmid → arxiv → title_author (guards: length > 20, ≤ 3 distinct work_ids),
then per-anchor legacy adoption (mag id from `ids`, pmh via `work_id_to_pmh_id_final`).

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.location_work_ids') AS target
USING (
  WITH t AS (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY provenance, native_id_namespace, native_id
             ORDER BY updated_date DESC) AS rwcnt
    FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
    QUALIFY rwcnt = 1
  ),
  pending AS (
    SELECT t.provenance, t.native_id_namespace, t.native_id, t.merge_key, t.ids
    FROM t
    LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
      ON  t.provenance = r.provenance
      AND t.native_id_namespace = r.native_id_namespace
      AND t.native_id = r.native_id
    WHERE r.native_id IS NULL OR r.work_id IS NULL
  ),
  d AS (
    SELECT doi AS k, MIN(work_id) AS v
    FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
    WHERE doi IS NOT NULL AND work_id IS NOT NULL GROUP BY doi
  ),
  p AS (
    SELECT pmid AS k, MIN(work_id) AS v
    FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
    WHERE pmid IS NOT NULL AND work_id IS NOT NULL GROUP BY pmid
  ),
  a AS (
    SELECT arxiv AS k, MIN(work_id) AS v
    FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
    WHERE arxiv IS NOT NULL AND work_id IS NOT NULL GROUP BY arxiv
  ),
  ta AS (
    SELECT title_author AS k, MIN(work_id) AS v
    FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
    WHERE title_author IS NOT NULL AND work_id IS NOT NULL
    GROUP BY title_author
    HAVING COUNT(DISTINCT work_id) <= 3
  ),
  pmh_mapping AS (
    SELECT LOWER(pmh_id) AS pmh_id, MIN(work_id) AS legacy_work_id
    FROM openalex.works_poc.work_id_to_pmh_id_final
    GROUP BY pmh_id
  ),
  resolved AS (
    SELECT
      pd.provenance, pd.native_id_namespace, pd.native_id,
      COALESCE(
        array_min(filter(array(d.v, p.v, a.v), x -> x IS NOT NULL)),
        CASE WHEN LENGTH(pd.merge_key.title_author) > 20 THEN ta.v END
      ) AS base_work_id,
      CASE
        WHEN d.v IS NOT NULL OR p.v IS NOT NULL OR a.v IS NOT NULL THEN
          CASE array_min(filter(array(d.v, p.v, a.v), x -> x IS NOT NULL))
            WHEN d.v THEN 'doi' WHEN p.v THEN 'pmid' ELSE 'arxiv' END
        WHEN LENGTH(pd.merge_key.title_author) > 20 AND ta.v IS NOT NULL THEN 'title_author'
      END AS base_source,
      CAST(get(filter(pd.ids, x -> x.namespace = 'mag').id, 0) AS BIGINT) AS mag_id
    FROM pending pd
    LEFT JOIN d  ON pd.merge_key.doi = d.k
    LEFT JOIN p  ON pd.merge_key.pmid = p.k
    LEFT JOIN a  ON pd.merge_key.arxiv = a.k
    LEFT JOIN ta ON pd.merge_key.title_author = ta.k
  )
  SELECT
    rs.provenance, rs.native_id_namespace, rs.native_id,
    CASE
      WHEN rs.provenance = 'mag' AND rs.base_work_id > 6600000000
           AND rs.mag_id IS NOT NULL THEN rs.mag_id
      WHEN rs.provenance IN ('repo', 'repo_backfill') AND rs.base_work_id > 6600000000
           AND pm.legacy_work_id IS NOT NULL THEN pm.legacy_work_id
      ELSE rs.base_work_id
    END AS work_id,
    CASE
      WHEN rs.provenance = 'mag' AND rs.base_work_id > 6600000000
           AND rs.mag_id IS NOT NULL THEN 'mag_legacy'
      WHEN rs.provenance IN ('repo', 'repo_backfill') AND rs.base_work_id > 6600000000
           AND pm.legacy_work_id IS NOT NULL THEN 'pmh_legacy'
      ELSE rs.base_source
    END AS work_id_source
  FROM resolved rs
  LEFT JOIN pmh_mapping pm
    ON rs.provenance IN ('repo', 'repo_backfill') AND LOWER(rs.native_id) = pm.pmh_id
) AS source
ON  target.provenance = source.provenance
AND target.native_id_namespace = source.native_id_namespace
AND target.native_id = source.native_id
WHEN MATCHED AND target.work_id IS NULL AND source.work_id IS NOT NULL
THEN UPDATE SET
  target.work_id = source.work_id,
  target.work_id_source = source.work_id_source,
  target.openalex_updated_dt = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  provenance, native_id_namespace, native_id, work_id, work_id_source,
  openalex_created_dt, openalex_updated_dt, seeded_dt, seeded_from
) VALUES (
  source.provenance, source.native_id_namespace, source.native_id,
  source.work_id, source.work_id_source,
  current_date(), current_timestamp(), NULL, NULL
)

In [0]:
SELECT format_number(COUNT(*), 0) AS registry_anchors,
       format_number(COUNT(work_id), 0) AS with_work_id
FROM identifier('openalex' || :env_suffix || '.works.location_work_ids')